# BOT-2 Shape Scoring · Helper Runtime 2

## 1. Setup

In [ ]:
%pip install -q -U timm "transformers>=4.56" safetensors

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, importlib
import pandas as pd

DRIVE_DIR = "/content/drive/MyDrive/Evaluation by prof. nam's research group"
FINAL_DIR = os.path.join(DRIVE_DIR, 'final')
ZIP_PATH = os.path.join(DRIVE_DIR, 'Shapes.zip')
N_WORKERS, WORKER = 3, 2          # helper notebook 2 of 2; the main notebook is worker 0
SAVE_WEIGHTS = 'ensemble'         # weight files kept on Drive: 'ensemble' (DPE networks, about 4 GB), 'all' or 'none'
SHUTDOWN_AT_END = True

sys.path.insert(0, FINAL_DIR)
import bot2score
from bot2score import utils, data, models, training, evaluation, baselines, figures, verify
for module in (utils, data, models, training, evaluation, baselines, figures, verify):
    importlib.reload(module)
utils.quiet_libraries()
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.precision', 3)

P = utils.make_paths(os.path.join(FINAL_DIR, 'results'))
log = utils.Log(os.path.join(P.logs, f'run_log_worker{WORKER}.txt'))
if SHUTDOWN_AT_END:
    utils.release_runtime_on_error(log)
env = utils.environment()
utils.save_json(os.path.join(P.logs, f'environment_worker{WORKER}.json'),
                {**env, 'bot2score': bot2score.__version__,
                 'code_sha256': utils.code_hashes(os.path.dirname(bot2score.__file__))})
log(f"bot2score {bot2score.__version__} | {env.get('gpu', 'no GPU')} | torch {env['torch']} | "
    f"timm {env['timm']} | transformers {env['transformers']}")

## 2. Data

In [ ]:
D = data.prepare(ZIP_PATH, P, log=log)

## 3. Training

In [ ]:
training.run(D, P, worker=WORKER, n_workers=N_WORKERS, save=SAVE_WEIGHTS, log=log)

## 4. Release the runtime

In [ ]:
if SHUTDOWN_AT_END:
    utils.release_runtime(log)